# Human-Prompted vs LLM-Prompted Trajectories

**Research question:** Do trajectories produced from human-written task prompts
(adapted from existing benchmarks) differ stylistically from trajectories
produced from LLM-generated task prompts (skill-based synthesis)?

**Important framing:** Both classes have AI-generated *trajectories* (DeepSeek-V3.2
is the teacher model in both cases per the Nemotron-Terminal paper). The only
difference is the *origin of the task prompt* — human-written benchmark prompts
vs LLM-synthesized prompts. So this is not a synthetic-vs-real text detector;
it's a study of how prompt source affects trajectory style.

**Experiments in this notebook:**
1. DistilBERT on full assistant text (random split) — likely topic-dominated
2. DistilBERT on function-words-only (random split) — headline "style" result
3. Within-class control: 3-way classifier on adapter domains (code/math/swe)
4. Domain-holdout split: train on some sub-domains, test on held-out ones


## 1. Install dependencies

In [1]:
!pip install datasets pandas -q
!pip install -U datasets pyarrow -q
!pip -q install huggingface_hub
!pip install scikit-learn -q
!pip install transformers torch accelerate -q


## 2. Load data

We load **all three adapter shards** (code, math, swe) for the human-prompted
class, and all three skill-based configs (easy, medium, mixed) for the
LLM-prompted class. We track sub-domain in a `subset` column so we can do
domain-based controls and splits later.


In [2]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import pandas as pd
import pyarrow.parquet as pq

# --- Human-prompted (dataset_adapters): load all three shards ---
adapter_shards = ["code.parquet", "math.parquet", "swe.parquet"]
SAMPLES_PER_ADAPTER = 400  # ~1200 total human-prompted

adapter_dfs = []
for shard in adapter_shards:
    local_path = hf_hub_download(
        repo_id="nvidia/Nemotron-Terminal-Corpus",
        filename=f"dataset_adapters/{shard}",
        repo_type="dataset"
    )
    pf = pq.ParquetFile(local_path)
    first_batch = next(pf.iter_batches(batch_size=SAMPLES_PER_ADAPTER))
    df = pd.DataFrame(first_batch.to_pylist())
    df["subset"] = shard.replace(".parquet", "")  # 'code', 'math', or 'swe'
    df["label"]  = "human_prompted"
    adapter_dfs.append(df)
    print(f"Loaded {shard}: {len(df)} rows")

human_prompted_df = pd.concat(adapter_dfs, ignore_index=True)

# --- LLM-prompted (skill_based_*): load all three configs ---
skill_configs = ["skill_based_easy", "skill_based_medium", "skill_based_mixed"]
SAMPLES_PER_SKILL = 400  # ~1200 total llm-prompted (matched to human side)

skill_dfs = []
for config in skill_configs:
    ds = load_dataset("nvidia/Nemotron-Terminal-Corpus", config)
    sample_ds = ds["train"].select(range(min(SAMPLES_PER_SKILL, len(ds["train"]))))
    df = sample_ds.to_pandas()
    df["subset"] = config
    df["label"]  = "llm_prompted"
    skill_dfs.append(df)
    print(f"Loaded {config}: {len(df)} rows")

llm_prompted_df = pd.concat(skill_dfs, ignore_index=True)

# Keep only columns that exist in both, plus our metadata
common_cols = list(set(human_prompted_df.columns) & set(llm_prompted_df.columns))
combined_df = pd.concat(
    [human_prompted_df[common_cols], llm_prompted_df[common_cols]],
    ignore_index=True
)

print(f"\nTotal rows: {len(combined_df)}")
print("\nLabel breakdown:")
print(combined_df["label"].value_counts())
print("\nSubset breakdown:")
print(combined_df["subset"].value_counts())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded code.parquet: 400 rows
Loaded math.parquet: 400 rows
Loaded swe.parquet: 400 rows
Loaded skill_based_easy: 400 rows
Loaded skill_based_medium: 400 rows
Loaded skill_based_mixed: 400 rows

Total rows: 2400

Label breakdown:
label
human_prompted    1200
llm_prompted      1200
Name: count, dtype: int64

Subset breakdown:
subset
code                  400
math                  400
swe                   400
skill_based_easy      400
skill_based_medium    400
skill_based_mixed     400
Name: count, dtype: int64


## 3. Extract assistant text from trajectories

In [3]:
import numpy as np

def get_assistant_text(row):
    convos = row["conversations"]
    if isinstance(convos, np.ndarray):
        convos = convos.tolist()
    assistant_turns = [
        t.get("content", "")
        for t in convos
        if isinstance(t, dict) and t.get("role") == "assistant"
    ]
    return " ".join(assistant_turns)

combined_df["assistant_text"] = combined_df.apply(get_assistant_text, axis=1)
print("Done. Sample lengths:")
print(combined_df["assistant_text"].str.len().describe())


Done. Sample lengths:
count      2400.000000
mean      40577.804583
std       25980.485766
min        1537.000000
25%       22729.750000
50%       35559.500000
75%       50311.250000
max      159558.000000
Name: assistant_text, dtype: float64


## 4. Build function-words-only version

Strips content vocabulary so the classifier can't lean on topic words like
`csv`, `json`, `dp`, `edges`. Whatever signal remains is closer to genuine
stylistic difference.


In [4]:
import re

FUNCTION_WORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then", "else",
    "when", "while", "for", "to", "of", "in", "on", "with", "by",
    "from", "as", "at", "is", "are", "was", "were", "be", "been",
    "being", "it", "this", "that", "these", "those", "we", "i",
    "you", "they", "he", "she", "will", "would", "can", "could",
    "should", "may", "might", "must", "not", "so", "because",
    "therefore", "however", "also", "first", "next", "finally",
    "now", "let", "need", "want", "check"
}

def keep_function_words_only(text):
    tokens = re.findall(r"\b\w+\b", text.lower())
    kept = [tok for tok in tokens if tok in FUNCTION_WORDS]
    return " ".join(kept)

combined_df["function_text"] = combined_df["assistant_text"].apply(keep_function_words_only)

# Drop rows that ended up with too little function-word content
combined_df = combined_df[combined_df["function_text"].str.len() > 20].reset_index(drop=True)
print(f"After filtering: {len(combined_df)} rows")
print("\nFunction-text length stats:")
print(combined_df["function_text"].str.len().describe())


After filtering: 2400 rows

Function-text length stats:
count     2400.000000
mean      7379.472500
std       5943.245796
min        201.000000
25%       3458.250000
50%       5383.000000
75%       9420.000000
max      44839.000000
Name: function_text, dtype: float64


## 5. Balance classes

Downsample to the smaller class so accuracy is meaningful. We balance the
two top-level labels but preserve the subset distribution within each.


In [5]:
human_df = combined_df[combined_df["label"] == "human_prompted"]
llm_df   = combined_df[combined_df["label"] == "llm_prompted"]

n = min(len(human_df), len(llm_df))
print(f"Balancing to {n} per class")

balanced_df = pd.concat([
    human_df.sample(n=n, random_state=42),
    llm_df.sample(n=n, random_state=42)
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

label_map = {"human_prompted": 0, "llm_prompted": 1}
balanced_df["y"] = balanced_df["label"].map(label_map)

print("\nBalanced label counts:")
print(balanced_df["label"].value_counts())
print("\nBalanced subset counts:")
print(balanced_df["subset"].value_counts())


Balancing to 1200 per class

Balanced label counts:
label
llm_prompted      1200
human_prompted    1200
Name: count, dtype: int64

Balanced subset counts:
subset
skill_based_easy      400
skill_based_mixed     400
swe                   400
skill_based_medium    400
math                  400
code                  400
Name: count, dtype: int64


## 6. DistilBERT setup

Reusable training and evaluation functions so we can run the same model on
multiple input columns (full text vs function-words) without code duplication.


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 512
BATCH_SIZE = 8
EPOCHS     = 3
LR         = 2e-5

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

def make_loader(texts, labels, shuffle=False):
    ds = TextDataset(texts.values, labels.values, tokenizer, MAX_LEN)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

def train_model(train_loader, dev_loader, num_labels=2, epochs=EPOCHS):
    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running_loss += out.loss.item()

        # Dev accuracy
        model.eval()
        dev_preds, dev_true = [], []
        with torch.no_grad():
            for batch in dev_loader:
                labels = batch.pop("labels")
                batch = {k: v.to(device) for k, v in batch.items()}
                logits = model(**batch).logits
                dev_preds.extend(logits.argmax(dim=-1).cpu().numpy())
                dev_true.extend(labels.numpy())
        dev_acc = accuracy_score(dev_true, dev_preds)
        print(f"Epoch {epoch+1}  train_loss={running_loss/len(train_loader):.4f}  dev_acc={dev_acc:.4f}")

    return model

def evaluate_model(name, model, loader, target_names=None, binary=True):
    model.eval()
    all_preds, all_probs, all_true = [], [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(labels.numpy())
            # Always grab full probability distribution
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

    import numpy as np
    all_probs = np.concatenate(all_probs, axis=0)
    acc = accuracy_score(all_true, all_preds)
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")

    if binary:
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_true, all_preds, average="binary", zero_division=0
        )
        auc = roc_auc_score(all_true, all_probs[:, 1])
    else:
        # Multi-class: use macro averaging so each class is weighted equally
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_true, all_preds, average="macro", zero_division=0
        )
        try:
            auc = roc_auc_score(all_true, all_probs, multi_class="ovr", average="macro")
        except ValueError:
            auc = float("nan")  # falls through if a class is missing from y_true

    print(f"Precision: {precision:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}")
    print()
    print(classification_report(all_true, all_preds,
                                target_names=target_names, zero_division=0))

    return {"name": name, "accuracy": acc, "precision": precision,
            "recall": recall, "f1": f1, "roc_auc": auc}


Device: cuda


## 7. Experiment 1 — Full text, random split

Baseline / ceiling: classifier sees everything including topic words. Expected
to be near 100%. The main purpose is to confirm separability and serve as a
reference for the function-words result.


In [7]:
from sklearn.model_selection import train_test_split

# Random 70/15/15 split, stratified by label
train_df, temp_df = train_test_split(
    balanced_df, test_size=0.30, random_state=42, stratify=balanced_df["y"]
)
dev_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["y"]
)

print(f"Train: {len(train_df)}  Dev: {len(dev_df)}  Test: {len(test_df)}")

# --- Full-text run ---
train_loader = make_loader(train_df["assistant_text"], train_df["y"], shuffle=True)
dev_loader   = make_loader(dev_df["assistant_text"],   dev_df["y"])
test_loader  = make_loader(test_df["assistant_text"],  test_df["y"])

print("\nTraining DistilBERT on full assistant_text...")
model_full = train_model(train_loader, dev_loader)

result_full_dev  = evaluate_model("Full text — Dev",  model_full, dev_loader,
                                   target_names=["human_prompted", "llm_prompted"])
result_full_test = evaluate_model("Full text — Test", model_full, test_loader,
                                   target_names=["human_prompted", "llm_prompted"])


Train: 1680  Dev: 360  Test: 360

Training DistilBERT on full assistant_text...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1  train_loss=0.1521  dev_acc=1.0000
Epoch 2  train_loss=0.0011  dev_acc=1.0000
Epoch 3  train_loss=0.0006  dev_acc=1.0000

=== Full text — Dev ===
Accuracy: 1.0000
Precision: 1.0000  Recall: 1.0000  F1: 1.0000  ROC-AUC: 1.0000

                precision    recall  f1-score   support

human_prompted       1.00      1.00      1.00       180
  llm_prompted       1.00      1.00      1.00       180

      accuracy                           1.00       360
     macro avg       1.00      1.00      1.00       360
  weighted avg       1.00      1.00      1.00       360


=== Full text — Test ===
Accuracy: 0.9972
Precision: 1.0000  Recall: 0.9944  F1: 0.9972  ROC-AUC: 1.0000

                precision    recall  f1-score   support

human_prompted       0.99      1.00      1.00       180
  llm_prompted       1.00      0.99      1.00       180

      accuracy                           1.00       360
     macro avg       1.00      1.00      1.00       360
  weighted avg       1.00      1.00  

## 8. Experiment 2 — Function-words only, random split (HEADLINE)

Strips topic vocabulary. Whatever accuracy remains is closer to genuine
stylistic separability. The gap between Experiment 1 and Experiment 2 is the
"topic vs style" decomposition.


In [8]:
train_loader_fw = make_loader(train_df["function_text"], train_df["y"], shuffle=True)
dev_loader_fw   = make_loader(dev_df["function_text"],   dev_df["y"])
test_loader_fw  = make_loader(test_df["function_text"],  test_df["y"])

print("Training DistilBERT on function_text only...")
model_fw = train_model(train_loader_fw, dev_loader_fw)

result_fw_dev  = evaluate_model("Function-words — Dev",  model_fw, dev_loader_fw,
                                 target_names=["human_prompted", "llm_prompted"])
result_fw_test = evaluate_model("Function-words — Test", model_fw, test_loader_fw,
                                 target_names=["human_prompted", "llm_prompted"])


Training DistilBERT on function_text only...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1  train_loss=0.2670  dev_acc=0.9833
Epoch 2  train_loss=0.0343  dev_acc=0.9944
Epoch 3  train_loss=0.0135  dev_acc=0.9889

=== Function-words — Dev ===
Accuracy: 0.9889
Precision: 0.9783  Recall: 1.0000  F1: 0.9890  ROC-AUC: 0.9996

                precision    recall  f1-score   support

human_prompted       1.00      0.98      0.99       180
  llm_prompted       0.98      1.00      0.99       180

      accuracy                           0.99       360
     macro avg       0.99      0.99      0.99       360
  weighted avg       0.99      0.99      0.99       360


=== Function-words — Test ===
Accuracy: 0.9889
Precision: 0.9835  Recall: 0.9944  F1: 0.9890  ROC-AUC: 0.9996

                precision    recall  f1-score   support

human_prompted       0.99      0.98      0.99       180
  llm_prompted       0.98      0.99      0.99       180

      accuracy                           0.99       360
     macro avg       0.99      0.99      0.99       360
  weighted avg       0.99  

## 9. Experiment 3 — Within-class control (3-way on adapter domains)

Trains a 3-way classifier on **only** the human-prompted samples (code vs math
vs swe), using `function_text`. This tests whether function words alone can
separate sub-domains *within* one class.

- If accuracy is near chance (~33%): function words are genuinely topic-neutral,
  and the headline result is measuring style.
- If accuracy is high: function words carry domain signal, and the headline
  result may still be partly topic-driven.


In [ ]:
# Within-class: only human_prompted, predict subset (code/math/swe)
within_df = balanced_df[balanced_df["label"] == "human_prompted"].copy()

subset_map = {"code": 0, "math": 1, "swe": 2}
within_df["y_sub"] = within_df["subset"].map(subset_map)

# Drop any subsets that don't fit (shouldn't happen, but safety)
within_df = within_df[within_df["y_sub"].notna()].copy()
within_df["y_sub"] = within_df["y_sub"].astype(int)

print("Within-class subset counts:")
print(within_df["subset"].value_counts())

w_train, w_temp = train_test_split(
    within_df, test_size=0.30, random_state=42, stratify=within_df["y_sub"]
)
w_dev, w_test = train_test_split(
    w_temp, test_size=0.50, random_state=42, stratify=w_temp["y_sub"]
)

w_train_loader = make_loader(w_train["function_text"], w_train["y_sub"], shuffle=True)
w_dev_loader   = make_loader(w_dev["function_text"],   w_dev["y_sub"])
w_test_loader  = make_loader(w_test["function_text"],  w_test["y_sub"])

print("\nTraining 3-way within-class classifier on function_text...")
model_within = train_model(w_train_loader, w_dev_loader, num_labels=3)

result_within_test = evaluate_model(
    "Within-class (function-words) — Test",
    model_within, w_test_loader,
    target_names=["code", "math", "swe"],
    binary=False
)


## 10. Experiment 4 — Domain-holdout split, function-words

Train on (math + code + easy + medium), test on (swe + mixed). If the model
still discriminates above chance on **held-out sub-domains**, the prompt-source
signal generalizes beyond the specific topics seen in training.


In [10]:
train_subsets = ["math", "code", "skill_based_easy", "skill_based_medium"]
test_subsets  = ["swe", "skill_based_mixed"]

holdout_train = balanced_df[balanced_df["subset"].isin(train_subsets)].copy()
holdout_test  = balanced_df[balanced_df["subset"].isin(test_subsets)].copy()

# Re-balance both halves so accuracy is meaningful
def balance(df):
    h = df[df["label"] == "human_prompted"]
    l = df[df["label"] == "llm_prompted"]
    n = min(len(h), len(l))
    return pd.concat([h.sample(n=n, random_state=42),
                      l.sample(n=n, random_state=42)],
                     ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

holdout_train = balance(holdout_train)
holdout_test  = balance(holdout_test)

# Carve a small dev set out of training for monitoring
holdout_train, holdout_dev = train_test_split(
    holdout_train, test_size=0.15, random_state=42, stratify=holdout_train["y"]
)

print(f"Holdout train: {len(holdout_train)} ({holdout_train['subset'].value_counts().to_dict()})")
print(f"Holdout dev:   {len(holdout_dev)}")
print(f"Holdout test:  {len(holdout_test)} ({holdout_test['subset'].value_counts().to_dict()})")

ho_train_loader = make_loader(holdout_train["function_text"], holdout_train["y"], shuffle=True)
ho_dev_loader   = make_loader(holdout_dev["function_text"],   holdout_dev["y"])
ho_test_loader  = make_loader(holdout_test["function_text"],  holdout_test["y"])

print("\nTraining DistilBERT on function_text with domain-holdout split...")
model_holdout = train_model(ho_train_loader, ho_dev_loader)

result_holdout_test = evaluate_model(
    "Domain-holdout (function-words) — Test",
    model_holdout, ho_test_loader,
    target_names=["human_prompted", "llm_prompted"]
)


Holdout train: 1360 ({'code': 344, 'skill_based_medium': 343, 'skill_based_easy': 337, 'math': 336})
Holdout dev:   240
Holdout test:  800 ({'skill_based_mixed': 400, 'swe': 400})

Training DistilBERT on function_text with domain-holdout split...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1  train_loss=0.2462  dev_acc=0.9833
Epoch 2  train_loss=0.0271  dev_acc=0.9958
Epoch 3  train_loss=0.0044  dev_acc=0.9958

=== Domain-holdout (function-words) — Test ===
Accuracy: 0.8350
Precision: 0.7519  Recall: 1.0000  F1: 0.8584  ROC-AUC: 0.9906

                precision    recall  f1-score   support

human_prompted       1.00      0.67      0.80       400
  llm_prompted       0.75      1.00      0.86       400

      accuracy                           0.83       800
     macro avg       0.88      0.83      0.83       800
  weighted avg       0.88      0.83      0.83       800



## 11. Summary table

The interesting comparison is across rows. If function-words is much lower than
full text, most of the original signal was topic. If domain-holdout is much
lower than the random split, the signal doesn't generalize beyond seen domains.
If the within-class control is near 33%, function words are topic-neutral.


In [11]:
summary = pd.DataFrame([
    result_full_test,
    result_fw_test,
    result_holdout_test,
    result_within_test,
])
cols = ["name", "accuracy", "precision", "recall", "f1", "roc_auc"]
summary = summary.reindex(columns=cols)
summary


,name,accuracy,precision,recall,f1,roc_auc
0,Full text — Test,0.997222,1.000000,0.994444,0.997214,1.000000
1,Function-words — Test,0.988889,0.983516,0.994444,0.988950,0.999630
2,Domain-holdout (function-words) — Test,0.835000,0.751880,1.000000,0.858369,0.990609
3,Within-class (function-words) — Test,0.944444,0.946426,0.944444,0.944357,0.993611


## How to interpret the result table

- **Full text accuracy near 100%, function-words much lower** → most original
  signal was topic vocabulary; style alone is harder to detect.
- **Full text and function-words both near 100%** → strong stylistic signal
  exists independent of topic words.
- **Domain-holdout drops sharply vs random split** → the classifier was learning
  domain-specific patterns that don't transfer.
- **Within-class 3-way classifier near chance (~33%)** → function words really
  are topic-neutral, supporting the headline interpretation.
- **Within-class 3-way classifier well above chance** → function words carry
  domain signal, so the main result is partly still topic.

A clean writeup combines all four numbers into a layered claim rather than
reporting any single one in isolation.


In [ ]:
## Saving the notebook to github

In [13]:
!pwd

/content
